In [6]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))  # add repo root
from rsr import rsr
import torch
import json

from ndtools import fun_binary_graph as fbg # ndtools available at github.com/jieunbyun/network-datasets
from ndtools.graphs import build_graph
from pathlib import Path
import networkx as nx   

In [7]:
DATASET = Path("data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs_bin.json").read_text(encoding="utf-8"))

# build base graph
G_base: nx.Graph = build_graph(nodes, edges, probs_dict)

In [8]:
#origin = 'n1'
origin = 'n52'
dests = ['n22', 'n66']
sys_upper_st = 1

def s_fun(comps_st):
    travel_time, sys_st, info = fbg.eval_travel_time_to_nearest(comps_st, G_base, origin, dests,
                                                         avg_speed=60, # km/h
                                                         target_max = [0.5, 0.25], # hours: it shouldn't take longer than this compared to the original travel time
                                                         length_attr = 'length_km')
    if sys_st >= sys_upper_st:
       path = info['path_filtered_edges'] 
       min_comps_st = {eid: ('>=', 1) for eid in path} # edges in the path are working
       min_comps_st['sys'] = ('>=', sys_st) # system edge is also working
    else:
        min_comps_st = None
    return travel_time, sys_st, min_comps_st

row_names = list(edges.keys()) 
n_state = 2 # binary states of components

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)


Check system function

In [9]:
comps_st = {eid: 1 for eid in edges.keys()}
travel_time, sys_st, info = s_fun(comps_st)
print(f"travel_time: {travel_time}, sys_st: {sys_st}, info: {info}")

travel_time: 0.7774131878455196, sys_st: 2, info: {'e0097': ('>=', 1), 'e0081': ('>=', 1), 'e0044': ('>=', 1), 'sys': ('>=', 2)}


In [10]:
result = rsr.run_ref_extraction_by_mcs(
    sfun=s_fun,
    probs=probs,
    row_names=row_names,
    n_state=n_state,
    output_dir="rsr_res",
    unk_prob_thres=5e-3,
    sys_upper_st=sys_upper_st
) 

---
Round: 1, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 0, Survival refs: 0, Failure refs: 0
Survival sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 2, System value: 0.7774131878455196. Total samples: 100000.
New ref (No. of conditions: 3): {'e0097': ('>=', 1), 'e0081': ('>=', 1), 'e0044': ('>=', 1), 'sys': ('>=', 1)}
Updated sys_vals: [0.777]
---
Round: 2, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 1, Survival refs: 1, Failure refs: 0
Survival sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 2, System value: 0.8902163337245933. Total samples: 100000.
New ref (No. of conditions: 4): {'e0106': ('>=', 1), 'e0098': ('>=', 1), 'e0081': ('>=', 1), 'e0044': ('>=', 1), 'sys': ('>=', 1)}
Updated sys_vals: [0.777, 0.89]
---
Round: 3, Unk. prob.: 2.709e-01
Upper probs: 7.291e-01, Lower probs: 0.000e+00
No. of 